# YOLOv8s AFPN + ATFL + Inner-IoU + DLQ Balanced + FTAL on Kaggle

这份 notebook 用于补充论文里的 `balanced + FTAL` 闭环实验。

核心目标：

- 保留 `AFPN + ATFL + Inner-IoU + DLQ balanced`
- 进一步叠加 `focused TAL`
- 验证 `balanced + FTAL` 是否优于现有 `balanced` 与其他 FTAL 分支

这份 notebook 默认采用你现有 balanced 分支的训练口径：

- `300 epochs`
- `imgsz=640`
- `batch=16`
- `workers=6`
- `device=0`
- `SGD`
- `seed=42`
- `patience=300`
- `save_period=10`
- `cos_lr=True`
- `close_mosaic=30`


In [ ]:
!nvidia-smi


In [ ]:
# 自动查找你上传到 Kaggle 的 3 类数据集根目录
from pathlib import Path
import yaml

INPUT_ROOT = Path('/kaggle/input')
candidate_roots = []
preferred_names = {
    'yolo-datasets-10k-yolov8-3class',
    'yolo-datasets-10k-yolov8-3cls',
    'yolo-datasets-10k-yolov8-3class-private',
}

for p in INPUT_ROOT.iterdir():
    if not p.is_dir():
        continue
    for q in [p] + [x for x in p.iterdir() if x.is_dir()]:
        if (q / 'images' / 'train').exists() and (q / 'labels' / 'train').exists():
            candidate_roots.append(q)

if not candidate_roots:
    raise FileNotFoundError('没有在 /kaggle/input 下找到包含 images/train 和 labels/train 的数据集目录。')


def score_root(path: Path):
    score = 0
    name = path.name.lower()
    if path.name in preferred_names:
        score += 100
    if '3class' in name or '3cls' in name:
        score += 10
    y = path / 'data.yaml'
    if y.exists():
        try:
            data = yaml.safe_load(y.read_text(encoding='utf-8')) or {}
            names = data.get('names', {})
            if isinstance(names, dict) and len(names) == 3:
                score += 20
        except Exception:
            pass
    return score

RUNTIME_ROOT = sorted(candidate_roots, key=score_root, reverse=True)[0]
print('RUNTIME_ROOT =', RUNTIME_ROOT)
print('candidates =')
for root in sorted(candidate_roots):
    print(' -', root)


In [ ]:
# 生成 Kaggle 训练用 data.yaml
from pathlib import Path

yaml_text = f'''path: {RUNTIME_ROOT.as_posix()}
train: images/train
val: images/val
test: images/test

nc: 3
names:
  0: chuck
  1: gripper
  2: drill_pipe
'''

DATA_CFG = Path('/kaggle/working/coalmine3_kaggle.yaml')
DATA_CFG.write_text(yaml_text, encoding='utf-8')
print(DATA_CFG.read_text(encoding='utf-8'))


In [ ]:
# 从 GitHub 拉取自定义源码，并以 editable 方式安装
from pathlib import Path
import shutil
import subprocess
import sys

REPO_URL = 'https://github.com/tuanziya666/yolov8s_kuangjing.git'
REPO_DIR = Path('/kaggle/working/yolov8s_kuangjing')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'ultralytics'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)

import ultralytics
print('python =', sys.executable)
print('repo_dir =', REPO_DIR)
print('ultralytics =', ultralytics.__file__)


In [ ]:
# 可选：清理旧结果
!rm -rf /kaggle/working/runs
!rm -rf /kaggle/working/evals
!rm -f /kaggle/working/yolov8s_afpn_atfl_inneriou_dlq_balanced_ftal_3cls_kaggle_results.zip


In [ ]:
# balanced + FTAL 的公共训练参数
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

EPOCHS = 300
IMGSZ = 640
BATCH = 16
WORKERS = 6
DEVICE = '0'
PATIENCE = 300
SAVE_PERIOD = 10
COS_LR = True
CLOSE_MOSAIC = 30
PROJECT = '/kaggle/working/runs'
NAME = 'yolov8s_afpn_atfl_inneriou_dlq_balanced_ftal_3cls_seed42_e300'

DLQ_LEVELS = 'p3p4'
DLQ_BASE_LEVEL = 3
DLQ_LAMBDA = 0.15
DLQ_SCORE_MODE = 'mul'
DLQ_ALPHA = 0.6
USE_DRILL_QUALITY_WEIGHT = True
DRILL_QUALITY_REFINE = True
DLQ_TARGET_CLASS_IDS = '2'
DRILL_QUALITY_BASE_WEIGHT = 1.15
DRILL_QUALITY_SMALL_H1 = 0.06
DRILL_QUALITY_SMALL_H2 = 0.09
DRILL_QUALITY_SMALL_W1 = 1.25
DRILL_QUALITY_SMALL_W2 = 1.10
FOCUSED_TAL_ENABLE = True
FOCUSED_TAL_TOPK = 24
FOCUSED_TAL_ALPHA = 1.0
FOCUSED_TAL_BETA = 2.9
FOCUSED_TAL_BOOST = 2.0
FOCUSED_TAL_TARGET_CLASS_IDS = '2'

print('DATA_CFG =', DATA_CFG)
print('EPOCHS =', EPOCHS)
print('IMGSZ =', IMGSZ)
print('BATCH =', BATCH)
print('WORKERS =', WORKERS)
print('DEVICE =', DEVICE)
print('PATIENCE =', PATIENCE)
print('SAVE_PERIOD =', SAVE_PERIOD)
print('COS_LR =', COS_LR)
print('CLOSE_MOSAIC =', CLOSE_MOSAIC)
print('NAME =', NAME)
print('DLQ_LAMBDA =', DLQ_LAMBDA)
print('USE_DRILL_QUALITY_WEIGHT =', USE_DRILL_QUALITY_WEIGHT)
print('FOCUSED_TAL_ENABLE =', FOCUSED_TAL_ENABLE)


In [ ]:
# 训练前清理显存
import gc
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('cuda ready =', torch.cuda.is_available())


In [ ]:
# 开始训练：AFPN + ATFL + Inner-IoU + DLQ balanced + FTAL
%cd /kaggle/working/yolov8s_kuangjing

import subprocess
import sys

TRAIN_SCRIPT = 'train_yolov8s_afpn_atfl_inneriou_dlq_head.py'
help_text = subprocess.check_output([sys.executable, TRAIN_SCRIPT, '--help'], text=True)
supports_focused_tal = '--focused-tal-enable' in help_text
print('supports_focused_tal_args =', supports_focused_tal)

cmd = [
    sys.executable,
    '-u',
    TRAIN_SCRIPT,
    '--data', str(DATA_CFG),
    '--model', 'ultralytics/cfg/models/v8/yolov8s_afpn.yaml',
    '--pretrained', 'yolov8s.pt',
    '--epochs', str(EPOCHS),
    '--batch', str(BATCH),
    '--imgsz', str(IMGSZ),
    '--workers', str(WORKERS),
    '--device', str(DEVICE),
    '--optimizer', 'SGD',
    '--seed', str(SEED),
    '--deterministic', 'True',
    '--cache', 'False',
    '--amp', 'True',
    '--patience', str(PATIENCE),
    '--save-period', str(SAVE_PERIOD),
    '--cos-lr', str(COS_LR),
    '--close-mosaic', str(CLOSE_MOSAIC),
    '--project', str(PROJECT),
    '--name', str(NAME),
    '--inner-ratio', '0.8',
    '--dlq-levels', str(DLQ_LEVELS),
    '--dlq-base-level', str(DLQ_BASE_LEVEL),
    '--dlq-lambda', str(DLQ_LAMBDA),
    '--dlq-score-mode', str(DLQ_SCORE_MODE),
    '--dlq-alpha', str(DLQ_ALPHA),
    '--use-drill-quality-weight', str(USE_DRILL_QUALITY_WEIGHT),
    '--drill-quality-refine', str(DRILL_QUALITY_REFINE),
    '--dlq-target-class-ids', str(DLQ_TARGET_CLASS_IDS),
    '--drill-quality-base-weight', str(DRILL_QUALITY_BASE_WEIGHT),
    '--drill-quality-small-h1', str(DRILL_QUALITY_SMALL_H1),
    '--drill-quality-small-h2', str(DRILL_QUALITY_SMALL_H2),
    '--drill-quality-small-w1', str(DRILL_QUALITY_SMALL_W1),
    '--drill-quality-small-w2', str(DRILL_QUALITY_SMALL_W2),
]

if supports_focused_tal:
    cmd += [
        '--focused-tal-enable', str(FOCUSED_TAL_ENABLE),
        '--focused-tal-topk', str(FOCUSED_TAL_TOPK),
        '--focused-tal-alpha', str(FOCUSED_TAL_ALPHA),
        '--focused-tal-beta', str(FOCUSED_TAL_BETA),
        '--focused-tal-boost', str(FOCUSED_TAL_BOOST),
        '--focused-tal-target-class-ids', str(FOCUSED_TAL_TARGET_CLASS_IDS),
    ]
else:
    raise RuntimeError('当前 GitHub 仓库里的训练脚本不支持 focused TAL 参数，请先同步最新代码后再跑。')

print('Running command:')
print(' '.join(cmd))

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end='')

ret = proc.wait()
if ret != 0:
    raise subprocess.CalledProcessError(ret, cmd)


In [ ]:
# 打包结果，便于从 Kaggle 下载
%cd /kaggle/working
!zip -r yolov8s_afpn_atfl_inneriou_dlq_balanced_ftal_3cls_kaggle_results.zip runs coalmine3_kaggle.yaml
!ls -lh /kaggle/working/yolov8s_afpn_atfl_inneriou_dlq_balanced_ftal_3cls_kaggle_results.zip
